# 04 · Breaking one GPU

Phases 1 to 3 sent requests one at a time. This phase sends **more and more at once** until the RTX 5090 can't keep up, and records exactly what gives way first and why. The result is the **baseline** every later phase has to beat.

All numbers are read from `results/04_breaking_one_gpu/`.

## How the load was applied

* **Locust**, closed loop: each simulated user sends its next request as soon as the last one finishes, so *users = requests in flight*.
* **Steps:** 1, 2, 4, 8, 16, 32, 64, 96, 128, 192, 256, 384, 512 users, 60 s each. The first 15 s of every step is skipped in the analysis, so each row describes the system after it settled.
* **Traffic mix:** the 24 prompts from `loadtest/prompts.json`, weighted like a chat product: 35% short questions, 25% multi-turn, 25% document summaries (~500 to 1,100 prompt tokens), 15% long answers (up to 768 output tokens). Every request gets a **unique tag** at the start of the user message, so repeated prompts can't be served from the prefix cache.
* **Engine:** the Phase 0 baseline config (`vllm/configs/baseline.yaml`), unchanged.
* **Gateway:** every harness on, except rate limits and load shedding, which are raised out of the way so the test measures the GPU and not the per-user limits.
* **Server side** numbers come from Prometheus over the same windows: requests running and waiting in vLLM, KV cache usage, preemptions, tokens per second, GPU utilization and power.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS = ROOT / "results" / "04_breaking_one_gpu"
sys.path.insert(0, str(ROOT / "loadtest"))
from results_logger import load_runs

runs = load_runs("04_breaking_one_gpu")
sweep = next((r for r in reversed(runs) if r["name"].startswith("load_sweep_")), None)
agents = next((r for r in reversed(runs) if r["name"] == "agent_load"), None)
print(f"load sweep: {sweep['name'] if sweep else None} | agent load: {bool(agents)}")
if sweep:
    env = sweep["environment"]
    gpu = env["gpus"][0] if env["gpus"] else {"name": "no GPU", "memory_mib": 0}
    print(f"hardware: {gpu['name']} | model: {sweep['config'].get('model')} | vLLM: {sweep['config'].get('vllm_image')} "
          f"| config: {sweep['config'].get('vllm_config')} | {sweep['metrics']['total_requests']:,} requests")

## Every step

In [ ]:
def v(d, *keys):
    for k in keys:
        d = (d or {}).get(k)
    return d

def cell(value, fmt, width, scale=1, unit=""):
    return ("-" if value is None else format(value * scale, fmt) + unit).rjust(width)

if not sweep:
    print("No load sweep yet: run scripts/run_phase4.sh on the GPU box, then copy results/ back.")
else:
    steps = sweep["metrics"]["steps"]
    print(f"{'users':>6}{'req/s':>8}{'out tok/s':>11}{'TTFT p50':>10}{'TTFT p95':>10}{'TTFT p99':>10}{'e2e p95':>9}"
          f"{'ITL p50':>9}{'running':>9}{'waiting':>9}{'KV max':>8}{'preempt':>9}{'GPU':>6}{'power':>7}{'errors':>8}")
    for s in steps:
        sv = s.get("server", {})
        print(cell(s["users"], "d", 6) + cell(s["requests_per_s"], ".2f", 8) + cell(sv.get("output_tokens_per_s"), ".0f", 11)
              + cell(s["ttft_s"]["p50"], ".0f", 10, 1000, "ms") + cell(s["ttft_s"]["p95"], ".0f", 10, 1000, "ms")
              + cell(s["ttft_s"]["p99"], ".0f", 10, 1000, "ms") + cell(s["e2e_s"]["p95"], ".1f", 9, 1, "s")
              + cell(s["itl_s"]["p50"], ".1f", 9, 1000, "ms") + cell(v(sv, "running", "avg"), ".0f", 9)
              + cell(v(sv, "waiting", "max"), ".0f", 9) + cell(v(sv, "kv_cache_usage", "max"), ".0%", 8)
              + cell(sv.get("preemptions"), ".0f", 9) + cell(v(sv, "gpu_utilization", "avg"), ".0%", 6)
              + cell(v(sv, "gpu_power_w", "avg"), ".0f", 7, 1, "W") + cell(s["errors"], "d", 8))

In [ ]:
if sweep:
    users = [s["users"] for s in steps]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    ax = axes[0][0]
    ax.plot(users, [v(s, "server", "output_tokens_per_s") for s in steps], "o-", label="output tokens/s")
    ax.set_title("Throughput"); ax.set_ylabel("output tokens / s")
    ax2 = ax.twinx(); ax2.plot(users, [s["requests_per_s"] for s in steps], "s--", color="#999", label="requests/s")
    ax2.set_ylabel("requests / s")

    ax = axes[0][1]
    for key, label in (("p50", "TTFT p50"), ("p95", "TTFT p95"), ("p99", "TTFT p99")):
        ax.plot(users, [s["ttft_s"][key] for s in steps], "o-", label=label)
    ax.set_yscale("log"); ax.set_title("Time to first token"); ax.set_ylabel("seconds (log)"); ax.legend()

    ax = axes[1][0]
    ax.plot(users, [v(s, "server", "running", "avg") for s in steps], "o-", label="running (avg)")
    ax.plot(users, [v(s, "server", "waiting", "max") for s in steps], "o-", label="waiting (max)")
    ax.set_title("Requests inside vLLM"); ax.legend()
    ax3 = ax.twinx(); ax3.plot(users, [v(s, "server", "kv_cache_usage", "max") for s in steps], "^:", color="#d94801", label="KV cache (max)")
    ax3.set_ylabel("KV cache used"); ax3.set_ylim(0, 1.05)

    ax = axes[1][1]
    ax.plot(users, [s["itl_s"]["p50"] and s["itl_s"]["p50"] * 1000 for s in steps], "o-", label="ITL p50")
    ax.plot(users, [s["itl_s"]["p95"] and s["itl_s"]["p95"] * 1000 for s in steps], "o-", label="ITL p95")
    ax.set_title("Time between tokens"); ax.set_ylabel("ms"); ax.legend()

    for ax in axes.flat:
        ax.set_xscale("log", base=2); ax.set_xlabel("concurrent users")
    plt.tight_layout(); plt.show()

## Where it broke

The first user count at which each kind of trouble showed up. `ttft_p95_over_slo` uses a 1 s target for time to first token.

In [ ]:
if sweep:
    for k, val in sweep["metrics"]["breaking_points"].items():
        print(f"{k:<28}{val}")

In [ ]:
if sweep:
    print("p95 by request type at each step (TTFT / end to end):")
    cats = sorted({c for s in steps for c in s["by_category"]})
    print(f"{'users':>6}" + "".join(f"{c:>26}" for c in cats))
    for s in steps:
        print(f"{s['users']:>6}" + "".join(
            cell(v(s, "by_category", c, "ttft_p95_s"), ".2f", 12, 1, "s") + " /" + cell(v(s, "by_category", c, "e2e_p95_s"), ".1f", 11, 1, "s")
            for c in cats))

## Is something other than the GPU the bottleneck?

Container CPU sampled every 10 s during the run (`docker stats`). The gateway is a single Python process that streams every token; if it maxes out a core before the GPU is busy, the GPU isn't the limit.

In [ ]:
stats_file = RESULTS / "baseline" / "container_stats.jsonl"
if sweep and stats_file.exists():
    rows = [json.loads(l) for l in stats_file.read_text().splitlines() if l.strip()]
    meta_start = sweep["load"]["start"]
    names = sorted({r["name"].replace("llm-inference-ops-", "").rsplit("-", 1)[0] for r in rows})
    print(f"{'users':>6}" + "".join(f"{n:>14}" for n in names) + "   (max CPU %, 100% = one core)")
    for s in steps:
        a, b = s["window"]
        in_win = [r for r in rows if a <= r["t"] < b]
        line = f"{s['users']:>6}"
        for n in names:
            vals = [float(r["cpu"].rstrip("%")) for r in in_win if n in r["name"]]
            line += cell(max(vals) if vals else None, ".0f", 13, 1, "%")
        print(line)
else:
    print("No container stats.")

## Concurrent agents

`scripts/agent_load.py` starts N CrewAI crews at the same moment (3 dependent LLM calls each), with a unique tag per task.

In [ ]:
if agents:
    print(f"{'crews':>6}{'task p50':>10}{'task p95':>10}{'tasks/min':>11}{'out tok/s':>11}{'failed':>8}   per agent p50")
    for lv in agents["metrics"]["levels"]:
        t, sv = lv["task_s"], lv.get("server") or {}
        print(f"{lv['concurrent_crews']:>6}{cell(t.get('p50'), '.1f', 9, 1, 's')}{cell(t.get('p95'), '.1f', 9, 1, 's')}"
              f"{lv['tasks_per_minute']:>11.1f}{cell(sv.get('output_tokens_per_s'), '.0f', 11)}{lv['failed']:>8}   "
              + ", ".join(f"{a.split()[0].lower()} {s:.1f}s" for a, s in lv["per_agent_p50_s"].items() if s))
else:
    print("No agent load run yet.")

## Grafana and Locust

*(added after the GPU session)*

## What happened

*(written after the GPU session, from the numbers above)*

## Phase 4 exit criteria

- [x] Locust step load test with per-request client measurements and server metrics per step
- [x] Concurrent CrewAI crews as a second, realistic load
- [x] Automatic breaking-point detection: queueing, KV cache full, preemptions, TTFT over target, errors
- [ ] Phase 4 session on the RTX 5090, 1 to 512 users
- [ ] Breaking point documented with its root cause